# LBPH Training & Evaluation

In [ ]:
# === IMPORTS ===
import os
import cv2
import gc
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# === CONFIGURATION ===

DATA_FOLDER               = os.path.join("..", "data")
AUGMENTED_FOLDER          = os.path.join(DATA_FOLDER, "7_augmented")
FINAL_PROCESSED_FOLDER    = os.path.join(DATA_FOLDER, "7_final_processed")
TEST_RAW_FOLDER           = os.path.join(DATA_FOLDER, "test_raw")
TEST_PROCESSED_FOLDER     = os.path.join(DATA_FOLDER, "7_test_processed")
PREVIEW_FOLDER            = os.path.join(DATA_FOLDER, "preview")
PREVIEW_EVALUATION_FOLDER = os.path.join(PREVIEW_FOLDER, "evaluation_results")
MODELS_FOLDER             = os.path.join("..", "models")

OUTPUT_SIZE   = (128, 128)   # Must match the size used in ImageProcessingPipeline
IMAGE_QUALITY = 95           # JPG save quality

os.makedirs(PREVIEW_EVALUATION_FOLDER, exist_ok=True)
os.makedirs(MODELS_FOLDER, exist_ok=True)

print("Configuration loaded!")
print(f"   AUGMENTED_FOLDER:       {AUGMENTED_FOLDER}")
print(f"   FINAL_PROCESSED_FOLDER: {FINAL_PROCESSED_FOLDER}")
print(f"   TEST_RAW_FOLDER:        {TEST_RAW_FOLDER}")
print(f"   TEST_PROCESSED_FOLDER:  {TEST_PROCESSED_FOLDER}")
print(f"   MODELS_FOLDER:          {MODELS_FOLDER}")
print(f"   OUTPUT_SIZE:            {OUTPUT_SIZE}")


# prepare training data

In [ ]:
def prepare_training_data(source_folder=None, test_ratio=0.2, random_seed=42, use_augmented=True, test_folder=None):
    """
    Prepare LBPH dataset without internal random split.

    Workflow:
    - If use_augmented=True: training data comes from `7_augmented/` by default.
    - If use_augmented=False: training data is forced to `7_final_processed/`.
    - Test data defaults to `7_test_processed/` (already preprocessed in ImageProcessingPipeline).
    - No train/test random split is performed in this notebook.

    Backward-compatible params kept:
    - test_ratio, random_seed are ignored in this mode.
    """
    _ = (test_ratio, random_seed)

    if source_folder is None:
        source_folder = AUGMENTED_FOLDER if use_augmented else FINAL_PROCESSED_FOLDER

    # Ensure the non-augmented branch always uses 7_final_processed for fair baseline.
    if not use_augmented and os.path.normpath(source_folder) != os.path.normpath(FINAL_PROCESSED_FOLDER):
        print(f"[i] use_augmented=False, overriding source folder to: {FINAL_PROCESSED_FOLDER}")
        source_folder = FINAL_PROCESSED_FOLDER

    if test_folder is None:
        test_folder = TEST_PROCESSED_FOLDER

    print("PREPARING TRAINING DATASET (NO INTERNAL SPLIT)")
    print("=" * 60)
    print(f"  Train source:     {source_folder}")
    print(f"  Test source:      {test_folder if test_folder else '(none)'}")
    print("  Test mode:        direct grayscale load (expects processed faces)")
    print(f"  Use augmentation: {'YES' if use_augmented else 'NO (final_processed baseline)'}\n")

    if not os.path.exists(source_folder):
        print(f"[X] Source folder not found: {source_folder}")
        return None

    person_dirs = sorted([
        d for d in os.listdir(source_folder)
        if os.path.isdir(os.path.join(source_folder, d))
    ])

    if not person_dirs:
        print("[X] No person folders found in training source!")
        return None

    label_names = []
    label_map = {}
    X_train, y_train = [], []
    X_test, y_test = [], []

    def preprocess_gray(img_gray):
        if img_gray is None:
            return None
        if img_gray.shape != OUTPUT_SIZE:
            img_gray = cv2.resize(img_gray, OUTPUT_SIZE)
        # final_processed/test_processed are already CLAHE-enhanced in ImageProcessingPipeline
        return img_gray

    def load_images(file_list, person_path, label):
        imgs, lbls = [], []
        failed = 0
        for f in file_list:
            file_path = os.path.join(person_path, f)
            img_gray = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            img_processed = preprocess_gray(img_gray)

            if img_processed is not None:
                imgs.append(img_processed)
                lbls.append(label)
            else:
                failed += 1
        return imgs, lbls, failed

    # Load training set from source_folder
    for label_id, person_name in enumerate(person_dirs):
        person_path = os.path.join(source_folder, person_name)
        label_map[person_name] = label_id
        label_names.append(person_name)

        all_files = sorted([
            f for f in os.listdir(person_path)
            if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}
        ])

        train_files = all_files

        if not train_files:
            print(f"  [!] {person_name}: no usable train files — skipped")
            continue

        tr_imgs, tr_lbls, tr_failed = load_images(train_files, person_path, label_id)
        X_train.extend(tr_imgs)
        y_train.extend(tr_lbls)

        print(f"  {person_name}: train={len(tr_imgs)}" + (f", failed={tr_failed}" if tr_failed else ""))

    # Optional fixed test set from external folder
    total_test_failed = 0
    if test_folder and os.path.exists(test_folder):
        print("\nLoading external test set...")
        for person_name, label_id in label_map.items():
            person_test_path = os.path.join(test_folder, person_name)
            if not os.path.exists(person_test_path):
                continue

            test_files = sorted([
                f for f in os.listdir(person_test_path)
                if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png'}
            ])

            te_imgs, te_lbls, te_failed = load_images(test_files, person_test_path, label_id)
            X_test.extend(te_imgs)
            y_test.extend(te_lbls)
            total_test_failed += te_failed

            if test_files:
                print(
                    f"  {person_name}: test={len(te_imgs)}"
                    + (f", failed={te_failed}" if te_failed else "")
                )
    else:
        print("\n[!] Test folder not found or not provided; evaluation set will be empty.")

    if not X_train:
        print("[X] Not enough training images!")
        return None

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test) if len(X_test) > 0 else np.array([])
    y_test = np.array(y_test) if len(y_test) > 0 else np.array([])

    stats = {
        "total_people": len(label_names),
        "train_size": len(X_train),
        "test_size": len(X_test),
        "test_failed": total_test_failed,
    }

    print(f"\nDATASET SUMMARY:")
    print(f"  Training set: {len(X_train)} images")
    print(f"  Testing set:  {len(X_test)} images")
    if total_test_failed > 0:
        print(f"  Test failed:  {total_test_failed} images (read failed)")
    print(f"  People:       {len(label_names)}")
    print(f"  Image size:   {OUTPUT_SIZE}")
    print("\n[OK] Training data ready!")

    return {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "label_names": label_names,
        "label_map": label_map,
        "stats": stats,
    }

# Prepare training dataset 
training_data = prepare_training_data()

# Train LBPH model

In [ ]:
def train_lbph_model(training_data, save_path=None,
                     radius=1, neighbors=8, grid_x=8, grid_y=8):
    """
    Train an LBPH face recognizer using the prepared training data.
    Evaluation is optional if external test data is available.
    """
    if training_data is None:
        print("[X] No training data provided!")
        return None
    
    if save_path is None:
        save_path = os.path.join(MODELS_FOLDER, "lbph_model.yml")
    
    print("TRAINING LBPH FACE RECOGNIZER")
    print("=" * 55)
    print(f"  Image size:  {OUTPUT_SIZE}")
    print(f"  Parameters:  radius={radius}, neighbors={neighbors}, grid=({grid_x}x{grid_y})")
    print(f"  Training on: {len(training_data['X_train'])} images")
    print(f"  Testing on:  {len(training_data['X_test'])} images")
    print(f"  People:      {len(training_data['label_names'])}\n")
    
    model = cv2.face.LBPHFaceRecognizer_create(
        radius=radius,
        neighbors=neighbors,
        grid_x=grid_x,
        grid_y=grid_y
    )
    
    print("Training model...")
    start_time = time.time()
    
    X_train = list(training_data["X_train"] )
    y_train = training_data["y_train"].astype(np.int32)
    model.train(X_train, y_train)
    
    train_time = time.time() - start_time
    print(f"[OK] Training complete in {train_time:.2f}s")

    X_test = training_data["X_test"]
    y_test = training_data["y_test"]
    label_names = training_data["label_names"]

    predictions = np.array([])
    confidences = np.array([])
    per_person_acc = {}
    accuracy = None

    if len(X_test) > 0:
        print("\nEvaluating on test set...")
        pred_list = []
        conf_list = []
        correct = 0

        for i in range(len(X_test)):
            pred_label, confidence = model.predict(X_test[i])
            pred_list.append(pred_label)
            conf_list.append(confidence)
            if pred_label == y_test[i]:
                correct += 1

        predictions = np.array(pred_list)
        confidences = np.array(conf_list)
        accuracy = correct / len(y_test) * 100 if len(y_test) > 0 else None

        print(f"\nEVALUATION RESULTS:")
        print(f"{'─' * 55}")
        print(f"  Overall accuracy: {accuracy:.1f}% ({correct}/{len(y_test)})")
        print(f"  Mean confidence:  {np.mean(confidences):.1f} (lower = better)")
        print(f"  Training time:    {train_time:.2f}s")

        print(f"\n  Per-person accuracy:")
        for lid, person_name in enumerate(label_names):
            mask = y_test == lid
            if np.sum(mask) > 0:
                person_correct = np.sum(predictions[mask] == lid)
                person_total = np.sum(mask)
                person_acc = person_correct / person_total * 100
                per_person_acc[person_name] = person_acc
                status = "[OK]" if person_acc >= 80 else ("[!]" if person_acc >= 50 else "[X]")
                print(f"    {status} {person_name}: {person_acc:.0f}% ({person_correct}/{person_total})")
    else:
        print("\n[!] No test set provided. Model trained without evaluation.")

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    model.save(save_path)
    print(f"\nModel saved: {save_path}")

    label_map_path = os.path.join(os.path.dirname(save_path), "label_map.txt")
    with open(label_map_path, "w") as f:
        for lid, name in enumerate(label_names):
            f.write(f"{lid},{name}\n")
    print(f"Label map saved: {label_map_path}")

    result = {
        "model": model,
        "accuracy": accuracy,
        "predictions": predictions,
        "confidences": confidences,
        "y_test": y_test,
        "label_names": label_names,
        "per_person_acc": per_person_acc,
        "train_time": train_time,
        "save_path": save_path,
    }

    return result

# Export baseline model: no augmentation -> lbph_model.yml
print("\n[1/2] Training NO-AUG model -> lbph_model.yml")
training_data_no_aug = prepare_training_data(
    source_folder=FINAL_PROCESSED_FOLDER,
    use_augmented=False,
    test_folder=TEST_PROCESSED_FOLDER,
 )
lbph_result_no_aug = train_lbph_model(
    training_data_no_aug,
    save_path=os.path.join(MODELS_FOLDER, "lbph_model.yml"),
)

# Export baseline model: with augmentation -> lbph_with_aug.yml
print("\n[2/2] Training WITH-AUG model -> lbph_with_aug.yml")
training_data = prepare_training_data(
    source_folder=AUGMENTED_FOLDER,
    use_augmented=True,
    test_folder=TEST_PROCESSED_FOLDER,
)
lbph_result = train_lbph_model(
    training_data,
    save_path=os.path.join(MODELS_FOLDER, "lbph_with_aug.yml"),
)

In [ ]:
def _build_balanced_subset(training_data, max_per_person=80, random_seed=42, strict_equal=True):
    """Build a class-balanced subset for tuning to reduce memory/time cost."""
    X_train = training_data["X_train"]
    y_train = training_data["y_train"]
    X_test = training_data["X_test"]
    y_test = training_data["y_test"]
    label_names = training_data["label_names"]

    rng = np.random.RandomState(random_seed)
    class_indices = {}
    for lid in np.unique(y_train):
        class_indices[int(lid)] = np.where(y_train == lid)[0]

    if not class_indices:
        return None

    if strict_equal:
        min_count = min(len(v) for v in class_indices.values())
        target_per_person = min(max_per_person, min_count)
    else:
        target_per_person = max_per_person

    subset_indices = []
    for lid, idx in class_indices.items():
        if strict_equal:
            chosen = rng.choice(idx, size=target_per_person, replace=False)
        else:
            take_n = min(target_per_person, len(idx))
            chosen = rng.choice(idx, size=take_n, replace=False)
        subset_indices.extend(chosen.tolist())

    subset_indices = np.array(sorted(subset_indices), dtype=np.int32)

    subset = {
        "X_train": X_train[subset_indices],
        "y_train": y_train[subset_indices],
        "X_test": X_test,
        "y_test": y_test,
        "label_names": label_names,
        "label_map": training_data["label_map"],
        "stats": dict(training_data.get("stats", {})),
    }
    subset["stats"]["train_size_subset"] = len(subset["X_train"] )
    subset["stats"]["target_per_person"] = int(target_per_person)
    subset["stats"]["strict_equal"] = bool(strict_equal)
    return subset


def _quick_eval_lbph(training_data, radius, neighbors, grid_x, grid_y):
    """Lightweight LBPH training/evaluation for tuning (no model save, no plots)."""
    model = cv2.face.LBPHFaceRecognizer_create(
        radius=radius,
        neighbors=neighbors,
        grid_x=grid_x,
        grid_y=grid_y,
    )

    start_time = time.time()
    X_train = list(training_data["X_train"] )
    y_train = training_data["y_train"].astype(np.int32)
    model.train(X_train, y_train)
    train_time = time.time() - start_time

    X_test = training_data["X_test"]
    y_test = training_data["y_test"]

    if len(X_test) == 0:
        return {
            "accuracy": -1,
            "mean_confidence": float("inf"),
            "train_time": train_time,
        }

    preds = []
    confs = []
    for img in X_test:
        pred_label, confidence = model.predict(img)
        preds.append(pred_label)
        confs.append(confidence)

    preds = np.array(preds)
    confs = np.array(confs, dtype=np.float32)
    acc = float(np.mean(preds == y_test) * 100.0)
    mean_conf = float(np.mean(confs)) if len(confs) > 0 else float("inf")

    return {
        "accuracy": acc,
        "mean_confidence": mean_conf,
        "train_time": train_time,
    }


def _evaluate_param_grid(training_data, param_grid, tag="coarse", lightweight=True, tune_model_path=None):
    """Evaluate one parameter grid; lightweight mode avoids save/extra overhead."""
    results = []
    print(f"\n{tag.upper()} GRID SEARCH")
    print("=" * 55)
    print(f"Mode: {'LIGHTWEIGHT' if lightweight else 'STANDARD'}")

    for p in param_grid:
        print(
            f"Trying r={p['radius']}, n={p['neighbors']}, grid={p['grid_x']}x{p['grid_y']} ..."
        )
        try:
            if lightweight:
                r = _quick_eval_lbph(
                    training_data,
                    radius=p["radius"],
                    neighbors=p["neighbors"],
                    grid_x=p["grid_x"],
                    grid_y=p["grid_y"],
                )
                acc = r["accuracy"]
                mean_conf = r["mean_confidence"]
                print(f"  -> acc={acc:.1f}%, mean_conf={mean_conf:.1f}, train={r['train_time']:.2f}s")
            else:
                result = train_lbph_model(
                    training_data,
                    radius=p["radius"],
                    neighbors=p["neighbors"],
                    grid_x=p["grid_x"],
                    grid_y=p["grid_y"],
                    save_path=tune_model_path or os.path.join(MODELS_FOLDER, "lbph_tune_temp.yml"),
                )
                acc = result["accuracy"] if result and result.get("accuracy") is not None else -1
                mean_conf = float(np.mean(result["confidences"])) if result is not None and len(result.get("confidences", [])) > 0 else float("inf")
                acc_txt = f"{acc:.1f}%" if acc >= 0 else "N/A"
                print(f"  -> acc={acc_txt}, mean_conf={mean_conf:.1f}")
                del result

            results.append((p, acc, mean_conf))
            gc.collect()

        except cv2.error as e:
            err = str(e)
            if "Insufficient memory" in err or "OutOfMemoryError" in err:
                print("  -> Skipped (Out of memory).")
            else:
                print(f"  -> Skipped (OpenCV error): {e}")
            results.append((p, -1, float("inf")))
            gc.collect()

    valid_results = [r for r in results if r[1] >= 0]
    if not valid_results:
        return None, results

    # Sort by higher accuracy, then lower mean confidence.
    valid_results.sort(key=lambda x: (-x[1], x[2]))
    best = valid_results[0]
    print(
        f"\n[OK] {tag} best: {best[0]} -> acc={best[1]:.1f}% / mean_conf={best[2]:.1f}"
    )
    return best, results


def _build_fine_grid(best_params):
    """Small local search around the coarse best point."""
    r0 = int(best_params["radius"] )
    n0 = int(best_params["neighbors"] )
    g0 = int(best_params["grid_x"] )

    radius_values = sorted(set([max(1, r0 - 1), r0, min(3, r0 + 1)]))
    neighbors_values = sorted(set([max(8, n0 - 4), n0, min(24, n0 + 4)]))
    if g0 == 8:
        grid_values = [8, 16]
    elif g0 == 16:
        grid_values = [8, 16]
    else:
        grid_values = [8, 16]

    fine_grid = []
    for r in radius_values:
        for n in neighbors_values:
            for g in grid_values:
                fine_grid.append({
                    "radius": r,
                    "neighbors": n,
                    "grid_x": g,
                    "grid_y": g,
                })
    return fine_grid


def tune_lbph_params(
    training_data,
    max_per_person=80,
    random_seed=42,
    strict_equal=True,
    lightweight=True,
    tune_temp_model_name="lbph_tune_temp.yml",
    final_model_name="lbph_final.yml",
):
    """
    Two-stage tuning:
    1) Coarse search on balanced subset
    2) Fine search around coarse best
    """
    if training_data is None:
        print("[X] No training data available for tuning.")
        return None

    subset_data = _build_balanced_subset(
        training_data,
        max_per_person=max_per_person,
        random_seed=random_seed,
        strict_equal=strict_equal,
    )
    if subset_data is None:
        print("[X] Failed to build balanced subset.")
        return None

    print("\nTUNING LBPH PARAMETERS (COARSE -> FINE)")
    print("=" * 60)
    print(f"Subset for tuning: {len(subset_data['X_train'])} / {len(training_data['X_train'])} train images")
    print(f"Balanced subset target per person: {subset_data['stats']['target_per_person']}")
    print(f"Strict equal sampling: {subset_data['stats']['strict_equal']}")
    print(f"Tuning mode: {'LIGHTWEIGHT' if lightweight else 'STANDARD'}")

    # Coarse range per your requested bounds.
    coarse_grid = [
        {"radius": r, "neighbors": n, "grid_x": g, "grid_y": g}
        for r in [1, 2, 3]
        for n in [8, 16, 24]
        for g in [8, 16]
    ]

    tune_temp_path = os.path.join(MODELS_FOLDER, tune_temp_model_name)
    coarse_best, _ = _evaluate_param_grid(
        subset_data,
        coarse_grid,
        tag="coarse",
        lightweight=lightweight,
        tune_model_path=tune_temp_path,
    )
    if coarse_best is None:
        print("[X] Coarse tuning failed. Try smaller max_per_person (e.g., 40).")
        return None

    fine_grid = _build_fine_grid(coarse_best[0])
    fine_best, _ = _evaluate_param_grid(
        subset_data,
        fine_grid,
        tag="fine",
        lightweight=lightweight,
        tune_model_path=tune_temp_path,
    )

    if fine_best is None:
        print("[!] Fine stage failed. Falling back to coarse best.")
        best_params = coarse_best[0]
        best_acc = coarse_best[1]
        best_conf = coarse_best[2]
    else:
        best_params = fine_best[0]
        best_acc = fine_best[1]
        best_conf = fine_best[2]

    print("\nFINAL TUNING RESULT")
    print("=" * 60)
    print(f"Best params: {best_params}")
    print(f"Best tuning acc: {best_acc:.1f}%")
    print(f"Best tuning mean_conf: {best_conf:.1f}")
    print(f"Final model name: {final_model_name}")

    return {
        "best_params": best_params,
        "final_model_name": final_model_name,
        "tune_temp_model_name": tune_temp_model_name,
    }


tuning_result = tune_lbph_params(
    training_data,
    max_per_person=80,
    random_seed=42,
    strict_equal=True,
    lightweight=True,
    tune_temp_model_name="lbph_tune_temp.yml",
    final_model_name="lbph_final.yml",
)

# Train one final tuned model on with-aug data -> lbph_final.yml
if tuning_result is not None:
    best_params = tuning_result["best_params"]
    final_model_path = os.path.join(MODELS_FOLDER, tuning_result["final_model_name"] )
    print("\nTraining FINAL tuned model -> lbph_final.yml")
    lbph_result = train_lbph_model(
        training_data,
        radius=best_params["radius"],
        neighbors=best_params["neighbors"],
        grid_x=best_params["grid_x"],
        grid_y=best_params["grid_y"],
        save_path=final_model_path,
    )

# Remove temporary tuning model file.
tune_temp_path = os.path.join(MODELS_FOLDER, "lbph_tune_temp.yml")
if os.path.exists(tune_temp_path):
    os.remove(tune_temp_path)
    print(f"Removed temporary model: {tune_temp_path}")

# LBPH Parameter Tuning (Grid Search)

# Model Evaluation: Confusion Matrix & Augmentation Impact

In [ ]:
def _evaluate_loaded_lbph_model(model_path, eval_data):
    """Evaluate a saved LBPH model on a prepared dataset without retraining."""
    if eval_data is None:
        print(f"[X] Evaluation data is None for model: {model_path}")
        return None

    if not os.path.exists(model_path):
        print(f"[X] Model file not found: {model_path}")
        return None

    X_test = eval_data["X_test"]
    y_test = eval_data["y_test"]
    label_names = eval_data["label_names"]

    if len(X_test) == 0:
        print(f"[X] Empty test set. Cannot evaluate model: {model_path}")
        return None

    model = cv2.face.LBPHFaceRecognizer_create()
    model.read(model_path)

    pred_list = []
    conf_list = []
    start_time = time.time()
    for i in range(len(X_test)):
        pred_label, confidence = model.predict(X_test[i])
        pred_list.append(pred_label)
        conf_list.append(confidence)
    infer_time = time.time() - start_time

    predictions = np.array(pred_list)
    confidences = np.array(conf_list)
    accuracy = float(np.mean(predictions == y_test) * 100.0)

    per_person_acc = {}
    for lid, person_name in enumerate(label_names):
        mask = y_test == lid
        if np.sum(mask) > 0:
            person_correct = np.sum(predictions[mask] == lid)
            person_total = np.sum(mask)
            per_person_acc[person_name] = float(person_correct / person_total * 100.0)

    return {
        "model": model,
        "accuracy": accuracy,
        "predictions": predictions,
        "confidences": confidences,
        "y_test": y_test,
        "label_names": label_names,
        "per_person_acc": per_person_acc,
        "train_time": 0.0,
        "infer_time": infer_time,
        "save_path": model_path,
    }


def plot_evaluation(lbph_result, save_dir=None):
    """
    Plot confusion matrix, per-person accuracy + confidence, and model summary metrics.
    """
    if lbph_result is None:
        print("[X] No model results to plot.")
        return

    if lbph_result.get("y_test") is None or len(lbph_result.get("y_test")) == 0:
        print("[!] No external test set available. Skipping evaluation plots.")
        print("    You can still use the trained model file for inference.")
        return

    if save_dir is None:
        save_dir = PREVIEW_EVALUATION_FOLDER
    os.makedirs(save_dir, exist_ok=True)

    y_test = lbph_result["y_test"]
    predictions = lbph_result["predictions"]
    confidences = lbph_result["confidences"]
    label_names = lbph_result["label_names"]
    per_person_acc = lbph_result["per_person_acc"]
    accuracy = lbph_result["accuracy"]

    report_dict = classification_report(
        y_test,
        predictions,
        labels=sorted(set(y_test)),
        target_names=[label_names[i][:12] for i in sorted(set(y_test))],
        zero_division=0,
        output_dict=True,
    )
    macro_p = float(report_dict.get("macro avg", {}).get("precision", 0.0))
    macro_r = float(report_dict.get("macro avg", {}).get("recall", 0.0))
    macro_f1 = float(report_dict.get("macro avg", {}).get("f1-score", 0.0))
    weighted_p = float(report_dict.get("weighted avg", {}).get("precision", 0.0))
    weighted_r = float(report_dict.get("weighted avg", {}).get("recall", 0.0))
    weighted_f1 = float(report_dict.get("weighted avg", {}).get("f1-score", 0.0))

    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    fig.suptitle(f"LBPH Model Evaluation - Overall Accuracy: {accuracy:.1f}%",
                 fontsize=14, fontweight='bold')

    cm = confusion_matrix(y_test, predictions, labels=list(range(len(label_names))))
    im = axes[0, 0].imshow(cm, interpolation='nearest', cmap='Blues')
    axes[0, 0].set_title("Confusion Matrix", fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel("Predicted")
    axes[0, 0].set_ylabel("Actual")

    short_names = [n[:8] for n in label_names]
    tick_positions = list(range(len(label_names)))
    axes[0, 0].set_xticks(tick_positions)
    axes[0, 0].set_yticks(tick_positions)
    axes[0, 0].set_xticklabels(short_names, rotation=45, ha='right', fontsize=7)
    axes[0, 0].set_yticklabels(short_names, fontsize=7)

    thresh = cm.max() / 2.0 if cm.size > 0 else 0.0
    for i in range(len(label_names)):
        for j in range(len(label_names)):
            color = "white" if cm[i, j] > thresh else "black"
            axes[0, 0].text(j, i, str(cm[i, j]), ha="center", va="center",
                           color=color, fontsize=7)

    fig.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

    persons = []
    accs = []
    mean_confs_by_person = []
    for lid, person_name in enumerate(label_names):
        mask = y_test == lid
        if np.sum(mask) == 0:
            continue
        persons.append(person_name)
        accs.append(float(np.mean(predictions[mask] == lid) * 100.0))
        mean_confs_by_person.append(float(np.mean(confidences[mask])))

    colors = ['#2ecc71' if a >= 80 else ('#f39c12' if a >= 50 else '#e74c3c') for a in accs]
    bars = axes[0, 1].barh(range(len(persons)), accs, color=colors)
    axes[0, 1].set_yticks(range(len(persons)))
    axes[0, 1].set_yticklabels([p[:12] for p in persons], fontsize=8)
    axes[0, 1].set_xlabel("Accuracy (%)")
    axes[0, 1].set_title("Per-Person Accuracy + Mean Confidence", fontsize=12, fontweight='bold')
    axes[0, 1].set_xlim(0, 120)
    axes[0, 1].axvline(x=80, color='gray', linestyle='--', alpha=0.5, label='80% target')
    axes[0, 1].legend(fontsize=8)
    axes[0, 1].grid(True, alpha=0.3, axis='x')

    for bar, acc, mc in zip(bars, accs, mean_confs_by_person):
        axes[0, 1].text(
            bar.get_width() + 1,
            bar.get_y() + bar.get_height() / 2,
            f"{acc:.0f}% | conf {mc:.1f}",
            va='center',
            fontsize=8,
        )

    correct_mask = predictions == y_test
    if np.sum(correct_mask) > 0:
        axes[1, 0].hist(confidences[correct_mask], bins=20, alpha=0.6,
                        color='#2ecc71', label='Correct', density=True)
    if np.sum(~correct_mask) > 0:
        axes[1, 0].hist(confidences[~correct_mask], bins=20, alpha=0.6,
                        color='#e74c3c', label='Incorrect', density=True)
    axes[1, 0].set_title("Confidence Score Distribution", fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel("Confidence (lower = more confident)")
    axes[1, 0].set_ylabel("Density")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].axis('off')
    summary_data = [
        ["Metric", "Value"],
        ["Overall Accuracy", f"{accuracy:.1f}%"],
        ["Macro Precision", f"{macro_p:.3f}"],
        ["Macro Recall", f"{macro_r:.3f}"],
        ["Macro F1", f"{macro_f1:.3f}"],
        ["Weighted Precision", f"{weighted_p:.3f}"],
        ["Weighted Recall", f"{weighted_r:.3f}"],
        ["Weighted F1", f"{weighted_f1:.3f}"],
        ["Total Test Images", f"{len(y_test)}"],
        ["Correct Predictions", f"{int(np.sum(correct_mask))}"],
        ["Wrong Predictions", f"{int(np.sum(~correct_mask))}"],
        ["Mean Confidence (Correct)", f"{np.mean(confidences[correct_mask]):.1f}" if np.sum(correct_mask) > 0 else "N/A"],
        ["Mean Confidence (Wrong)", f"{np.mean(confidences[~correct_mask]):.1f}" if np.sum(~correct_mask) > 0 else "N/A"],
        ["People with >= 80% Acc", f"{sum(1 for a in accs if a >= 80)}/{len(accs)}"],
        ["Inference Time", f"{lbph_result.get('infer_time', 0.0):.2f}s"],
    ]

    table = axes[1, 1].table(cellText=summary_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.5)
    for j in range(2):
        table[0, j].set_facecolor('#4472C4')
        table[0, j].set_text_props(color='white', fontweight='bold')
    axes[1, 1].set_title("Model Summary", fontsize=12, fontweight='bold', pad=20)

    plt.tight_layout()

    eval_path = os.path.join(save_dir, "lbph_evaluation.png")
    fig.savefig(eval_path, dpi=150, bbox_inches='tight')
    print(f"Evaluation plot saved: {eval_path}")
    plt.show()

    present_labels = sorted(set(y_test))
    present_names = [label_names[i][:12] for i in present_labels]
    print("\nCLASSIFICATION REPORT:")
    print(classification_report(
        y_test,
        predictions,
        labels=present_labels,
        target_names=present_names,
        zero_division=0,
    ))


def compare_augmentation_impact(test_folder=None):
    """
    Compare fixed saved models on the same test set:
    lbph_model / lbph_with_aug / lbph_final.
    """
    if test_folder is None:
        test_folder = TEST_PROCESSED_FOLDER

    print("\nCOMPARING AUGMENTATION IMPACT (SAVED MODELS)")
    print("=" * 65)
    print(f"Fixed test set: {test_folder}\n")

    noaug_data = prepare_training_data(
        source_folder=FINAL_PROCESSED_FOLDER,
        use_augmented=False,
        test_folder=test_folder,
    )
    aug_data = prepare_training_data(
        source_folder=AUGMENTED_FOLDER,
        use_augmented=True,
        test_folder=test_folder,
    )

    noaug_model_path = os.path.join(MODELS_FOLDER, "lbph_model.yml")
    aug_model_path = os.path.join(MODELS_FOLDER, "lbph_with_aug.yml")
    tuned_model_path = os.path.join(MODELS_FOLDER, "lbph_final.yml")

    missing_files = [p for p in [noaug_model_path, aug_model_path, tuned_model_path] if not os.path.exists(p)]
    if missing_files:
        print("[X] Missing required model files for 3-way comparison:")
        for p in missing_files:
            print(f"    - {p}")
        return None

    noaug_result = _evaluate_loaded_lbph_model(noaug_model_path, noaug_data)
    aug_result = _evaluate_loaded_lbph_model(aug_model_path, aug_data)
    tuned_result = _evaluate_loaded_lbph_model(tuned_model_path, aug_data)

    if noaug_result is None or aug_result is None or tuned_result is None:
        print("[X] Comparison could not be completed (missing model files or invalid data).")
        return None

    noaug_acc = noaug_result["accuracy"]
    aug_acc = aug_result["accuracy"]
    tuned_acc = tuned_result["accuracy"]

    improvement_aug_vs_noaug = aug_acc - noaug_acc
    improvement_tuned_vs_aug = tuned_acc - aug_acc

    print("\nAUGMENTATION + TUNING COMPARISON SUMMARY")
    print("-" * 60)
    print(f"No augmentation:        {noaug_acc:.1f}%")
    print(f"With augmentation:      {aug_acc:.1f}%  ({improvement_aug_vs_noaug:+.1f} vs no-aug)")
    print(f"With aug + tuned:       {tuned_acc:.1f}%  ({improvement_tuned_vs_aug:+.1f} vs with-aug)")

    labels = ["No Aug", "With Aug", "With Aug + Tuned"]
    values = [noaug_acc, aug_acc, tuned_acc]
    colors = ["#f39c12", "#2ecc71", "#3498db"]

    plt.figure(figsize=(8, 4.8))
    bars = plt.bar(labels, values, color=colors, alpha=0.9)
    plt.ylim(0, 100)
    plt.ylabel("Accuracy (%)")
    plt.title("LBPH Accuracy: No Aug vs With Aug vs With Aug + Tuned")
    plt.grid(axis='y', alpha=0.25)

    for b, v in zip(bars, values):
        plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha='center', fontsize=10)

    cmp_path = os.path.join(PREVIEW_EVALUATION_FOLDER, "lbph_aug_tune_comparison.png")
    plt.tight_layout()
    plt.savefig(cmp_path, dpi=150)
    print(f"Comparison plot saved: {cmp_path}")
    plt.show()

    return {
        "no_aug": noaug_result,
        "with_aug": aug_result,
        "with_aug_tuned": tuned_result,
        "improvement_aug_vs_noaug": improvement_aug_vs_noaug,
        "improvement_tuned_vs_aug": improvement_tuned_vs_aug,
    }


# Plot evaluation for the latest trained model
print("Plotting LBPH model evaluation...\n")
plot_evaluation(lbph_result)

In [ ]:
def print_confusion_details(lbph_result, top_k=20):
    """Print detailed confusion pairs sorted by error count."""
    if lbph_result is None or lbph_result.get("y_test") is None or len(lbph_result.get("y_test")) == 0:
        print("[X] No test predictions available for confusion details.")
        return []

    y_test = lbph_result["y_test"]
    predictions = lbph_result["predictions"]
    label_names = lbph_result["label_names"]

    cm = confusion_matrix(y_test, predictions, labels=list(range(len(label_names))))

    pairs = []
    for actual_idx in range(cm.shape[0]):
        for pred_idx in range(cm.shape[1]):
            if actual_idx == pred_idx:
                continue
            count = int(cm[actual_idx, pred_idx])
            if count > 0:
                pairs.append((count, actual_idx, pred_idx))

    pairs.sort(reverse=True, key=lambda x: x[0])

    print("\nTOP CONFUSION PAIRS")
    print("=" * 60)
    if not pairs:
        print("[OK] No off-diagonal confusions found.")
        return []

    for rank, (count, actual_idx, pred_idx) in enumerate(pairs[:top_k], start=1):
        actual_name = label_names[actual_idx]
        pred_name = label_names[pred_idx]
        print(f"  {rank:>2}. {actual_name} -> {pred_name}: {count}")

    print("\nPER-PERSON MISCLASSIFICATION BREAKDOWN")
    print("=" * 60)
    for actual_idx, actual_name in enumerate(label_names):
        row = cm[actual_idx]
        row_total = int(np.sum(row))
        correct = int(row[actual_idx])
        wrong = row_total - correct
        if row_total == 0:
            continue

        print(f"  {actual_name}: total={row_total}, correct={correct}, wrong={wrong}")
        if wrong > 0:
            conf_targets = []
            for pred_idx, cnt in enumerate(row):
                if pred_idx != actual_idx and cnt > 0:
                    conf_targets.append((int(cnt), label_names[pred_idx]))
            conf_targets.sort(reverse=True, key=lambda x: x[0])
            detail = ", ".join([f"{name}:{cnt}" for cnt, name in conf_targets])
            print(f"    confused_as -> {detail}")

    return pairs


confusion_pairs = print_confusion_details(lbph_result, top_k=20)

# Confusion Details (Who Confuses With Whom)

In [ ]:
CONFIDENCE_THRESHOLD = 160  # LBPH confidence: lower is better; default threshold raised for tuned models


def evaluate_confidence_thresholds(lbph_result, thresholds=(100, 120, 140, 160, 180, 200)):
    """Evaluate Unknown-rejection behavior under multiple confidence thresholds."""
    if lbph_result is None or lbph_result.get("y_test") is None or len(lbph_result.get("y_test")) == 0:
        print("[X] No test predictions available for threshold sweep.")
        return []

    y_test = lbph_result["y_test"]
    predictions = lbph_result["predictions"]
    confidences = lbph_result["confidences"]

    print("\nCONFIDENCE THRESHOLD SWEEP")
    print("=" * 60)
    print("(LBPH confidence: lower is better)")
    print(f"Confidence range: min={np.min(confidences):.1f}, mean={np.mean(confidences):.1f}, max={np.max(confidences):.1f}")

    rows = []
    for th in thresholds:
        accepted = confidences <= th
        rejected = ~accepted

        accepted_count = int(np.sum(accepted))
        rejected_count = int(np.sum(rejected))
        total = len(y_test)

        if accepted_count > 0:
            accepted_acc = float(np.mean(predictions[accepted] == y_test[accepted]) * 100)
        else:
            accepted_acc = 0.0

        overall_with_unknown = float(np.sum((predictions == y_test) & accepted) / total * 100)

        row = {
            "threshold": th,
            "accepted": accepted_count,
            "rejected": rejected_count,
            "rejected_pct": rejected_count / total * 100,
            "accepted_acc": accepted_acc,
            "overall_with_unknown": overall_with_unknown,
        }
        rows.append(row)

        print(
            f"  th={th:>3} | accepted={accepted_count:>3} | rejected={rejected_count:>3} "
            f"({row['rejected_pct']:.1f}%) | accepted-acc={accepted_acc:.1f}% | "
            f"overall-with-unknown={overall_with_unknown:.1f}%"
        )

    return rows


threshold_results = evaluate_confidence_thresholds(
    lbph_result,
    thresholds=(100, 120, 140, 160, 180, 200),
)

# Confidence Threshold Sweep

In [ ]:
# Compare fixed 3 saved models on the same processed test set (no retraining, no retuning)
comparison_results = compare_augmentation_impact(
    test_folder=TEST_PROCESSED_FOLDER,
)